In [ ]:
%load_ext autoreload
%autoreload 2

# Step 1: Load the cleaned dataset and prepare one commodity for forecasting
import sys
import pandas as pd
sys.path.insert(0, "..")
from src.forecasting import prepare_commodity_data

# Load our cleaned dataset
cleaned_df = pd.read_csv(
    "../data/processed/cleaned_farmgate_prices.csv",
    parse_dates=["Date"]
)

# Test the reusable function using Palay
palay_data = prepare_commodity_data(
    cleaned_df,
    "Cereals",
    "Palay [Paddy] Other Variety, dry (conv. to 14% mc)"
)

print("Total months:", len(palay_data))
print("Missing prices:", palay_data["Price"].isna().sum())

display(palay_data.head())

In [31]:
palay_data

,Geolocation,Category,Commodity,Price,Was_Interpolated
Date,,,,,
2010-01,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",14.82,False
2010-02,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",15.34,False
2010-03,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",15.58,False
2010-04,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",14.92,False
2010-05,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",14.99,False
...,...,...,...,...,...
2026-06,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",21.78,False
2026-07,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",22.39,False
2026-08,PHILIPPINES,Cereals,"Palay [Paddy] Other Variety, dry (conv. to 14%...",22.16,False


In [ ]:
# Step 2: Initialize and display the three approved forecasting models
# Import the reusable function from forecasting.py
from src.forecasting import create_forecasting_models

# Create the three approved forecasting models
models = create_forecasting_models()

# Display each model and its configuration
for model_name, model in models.items():
    print(f"{model_name}:")
    print(model)
    print()

In [30]:
# Step 3: Split the data chronologically and create six-month testing windows
from src.evaluation import prepare_train_test

# Split Palay prices into training and testing datasets
monthly_prices, train_data, test_data, test_windows = prepare_train_test(
    palay_data
)

# Check the sizes of each dataset
print("Total months:", len(monthly_prices))
print("Training months:", len(train_data))
print("Testing months:", len(test_data))
print("Six-month testing windows:", len(test_windows))

# Check the chronological dates
print("\nTraining period:", train_data.index.min(), "to", train_data.index.max())
print("Testing period:", test_data.index.min(), "to", test_data.index.max())

# Check the number of months in each testing window
print("Window sizes:", [len(window) for window in test_windows], "\n")
# Test windows
print(test_windows[0:2])

Total months: 202
Training months: 161
Testing months: 41
Six-month testing windows: 6

Training period: 2010-01 to 2023-05
Testing period: 2023-06 to 2026-10
Window sizes: [6, 6, 6, 6, 6, 6] 

[Date
2023-06    19.23
2023-07    19.44
2023-08    22.16
2023-09    19.91
2023-10    20.60
2023-11    21.95
Freq: M, Name: Price, dtype: float64, Date
2023-12    22.89
2024-01    25.01
2024-02    25.17
2024-03    24.64
2024-04    24.68
2024-05    24.95
Freq: M, Name: Price, dtype: float64]


In [32]:
# Step 4: Evaluate the Seasonal Naive 
from src.evaluation import backtest_model
import numpy as np

# Evaluate the Seasonal Naive baseline using six-month windows
seasonal_naive_results = backtest_model(
    models["Seasonal Naive"],
    palay_data,
    test_windows
)

# Display the evaluation results for each testing window
display(seasonal_naive_results)

#---------------------------------------------

# Average the MAE values from all six testing windows
overall_mae = seasonal_naive_results["MAE"].mean()

# Calculate overall RMSE using the squared window RMSE values
overall_rmse = np.sqrt(
    (seasonal_naive_results["RMSE"] ** 2).mean()
)

print("Seasonal Naive Overall Evaluation")
print("MAE:", round(overall_mae, 2))
print("RMSE:", round(overall_rmse, 2))

,Test Period,MAE,RMSE
0,2023-06 to 2023-11,3.066667,3.296999
1,2023-12 to 2024-05,6.218333,6.257615
2,2024-06 to 2024-11,2.788333,3.451857
3,2024-12 to 2025-05,5.173333,5.440144
4,2025-06 to 2025-11,6.286667,6.525317
5,2025-12 to 2026-05,3.340000,3.738052


Seasonal Naive Overall Evaluation
MAE: 4.48
RMSE: 4.97


In [ ]:
# Step 5: Evaluate Holt-Winters 

# Train and evaluate Holt-Winters across the same six testing windows
holt_winters_results = backtest_model(
    models["Holt-Winters"],
    palay_data,
    test_windows
)

# Display the MAE and RMSE for each testing window
display(holt_winters_results)

#---------------------------------

# Calculate the average MAE across all six testing windows
holt_overall_mae = holt_winters_results["MAE"].mean()

# Calculate overall RMSE across all six testing windows
holt_overall_rmse = np.sqrt(
    (holt_winters_results["RMSE"] ** 2).mean()
)

print("Holt-Winters Overall Evaluation")
print("MAE:", round(holt_overall_mae, 2))
print("RMSE:", round(holt_overall_rmse, 2))


In [28]:
# Step 6: Evaluate AutoARIMA 

# Evaluate AutoARIMA across the same six testing windows
autoarima_results = backtest_model(
    models["AutoARIMA"],
    palay_data,
    test_windows
)

# Display the evaluation results for each window
display(autoarima_results)

#-------------------------------------------

# Calculate overall MAE and RMSE across all six windows
autoarima_overall_mae = autoarima_results["MAE"].mean()
autoarima_overall_rmse = np.sqrt(
    (autoarima_results["RMSE"] ** 2).mean()
)

# Display the overall evaluation
print("AutoARIMA Overall Evaluation")
print("MAE:", round(autoarima_overall_mae, 2))
print("RMSE:", round(autoarima_overall_rmse, 2))

,Test Period,MAE,RMSE
0,2023-06 to 2023-11,1.395115,1.808165
1,2023-12 to 2024-05,2.371725,2.492884
2,2024-06 to 2024-11,2.243975,2.905560
3,2024-12 to 2025-05,1.226667,1.578574
4,2025-06 to 2025-11,1.202985,1.329284
5,2025-12 to 2026-05,4.457870,4.882352


AutoARIMA Overall Evaluation
MAE: 2.15
RMSE: 2.77


In [29]:
# Step 7: Compare all three models and select the best

# Organize the overall evaluation results into one table
model_comparison = pd.DataFrame([
    {"Model": "Seasonal Naive", "MAE": overall_mae, "RMSE": overall_rmse},
    {"Model": "Holt-Winters", "MAE": holt_overall_mae, "RMSE": holt_overall_rmse},
    {"Model": "AutoARIMA", "MAE": autoarima_overall_mae, "RMSE": autoarima_overall_rmse}
])

# Rank models by lowest MAE, then RMSE
model_comparison = model_comparison.sort_values(
    by=["MAE", "RMSE"]
).reset_index(drop=True)

# Display the comparison table
display(model_comparison.round(2))

# Select the model with the lowest forecasting error
best_model_name = model_comparison.iloc[0]["Model"]

print("Best Model for Palay:", best_model_name)

,Model,MAE,RMSE
0,Holt-Winters,2.12,2.65
1,AutoARIMA,2.15,2.77
2,Seasonal Naive,4.48,4.97


Best Model for Palay: Holt-Winters


In [18]:
# Step 9: Train the best model and generate six-month future forecasts
from src.forecasting import fit_and_forecast

# Train the selected model
fitted_model, future_forecast = fit_and_forecast(
    models[best_model_name],
    monthly_prices,
    forecast_months=6
)

# Display the last observed date and the selected model
print("Best Model:", best_model_name)
print("Last Observed Date:", monthly_prices.index.max())

# Display the six predicted future prices
print("\nSix-Month Future Forecast:")
display(future_forecast.round(2))

Best Model: Holt-Winters
Last Observed Date: 2026-10

Six-Month Future Forecast:


2026-11    21.47
2026-12    21.84
2027-01    22.27
2027-02    22.65
2027-03    22.74
2027-04    22.84
Freq: M, Name: Price, dtype: float64

In [ ]:
# Step 10: Organize the six-month forecasts with model evaluation results

# Get the evaluation metrics of the selected model
best_metrics = model_comparison[
    model_comparison["Model"] == best_model_name
].iloc[0]

# Create a table containing the forecast details
forecast_table = pd.DataFrame({
    "Category": palay_data["Category"].iloc[0],
    "Commodity": palay_data["Commodity"].iloc[0],
    "Last Observed Date": str(monthly_prices.index[-1]),
    "Last Observed Price": monthly_prices.iloc[-1],
    "Forecast Date": future_forecast.index.astype(str),
    "Forecast Price": future_forecast.to_numpy(),
    "Model": best_model_name,
    "MAE": best_metrics["MAE"],
    "RMSE": best_metrics["RMSE"]
})

# Display the forecast table with two decimal places
display(forecast_table.round(2))


In [23]:
# Step 11: Save the fitted model using joblib
import joblib
from pathlib import Path

# Create a folder for saved forecasting models
models_folder = Path("../models")
models_folder.mkdir(parents=True, exist_ok=True)

# Save the trained Palay model
model_path = models_folder / "palay_holt_winters.joblib"
joblib.dump(fitted_model, model_path)

print("Model saved successfully:", model_path)

Model saved successfully: ..\models\palay_holt_winters.joblib


In [ ]:
# Step 12: Load the saved model and verify its forecasts
import numpy as np

# Load the previously trained Holt-Winters model
loaded_model = joblib.load(model_path)

# Generate six-month forecasts without retraining
loaded_forecast = loaded_model.predict(fh=range(1, 7))

# Display the predictions from the loaded model
display(loaded_forecast.round(2))


In [27]:
# Step 13: Verify the saved model produces the same predictions

# Compare the original forecasts with the loaded model's forecasts
forecasts_match = np.allclose(
    future_forecast.to_numpy(),
    loaded_forecast.to_numpy()
)

print("Forecasts Match:", forecasts_match)
print("Loaded Model Is Fitted:", loaded_model.is_fitted)


Forecasts Match: True
Loaded Model Is Fitted: True
